# In-Context Learning: Emergence of Meta-Learning in Large Language Models

## Learning Objectives

1. Understand how transformers implement in-context learning without parameter updates
2. Implement ICL simulators and analyze few-shot vs. zero-shot performance
3. Compare ICL with fine-tuning in terms of efficiency, accuracy, and task switching
4. Design effective prompts and analyze robustness to example variations

In [ ]:
# Core imports and device setup
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
from collections import defaultdict
import warnings

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Level 1: Basic In-Context Learning Simulation
# Demonstrates core concept: pattern matching without weight updates

class SimpleICLSimulator:
    """
    Simulates ICL by matching test input to examples.
    Core idea: pattern recognition from examples without parameter updates.
    """
    
    def __init__(self, embedding_dim=16):
        self.embedding_dim = embedding_dim
        self.embeddings = {}  # Frozen embeddings (no updates during inference)
    
    def _embed_text(self, text):
        """Convert text to embedding (fixed, not trained)."""
        tokens = text.lower().split()
        embedding = np.zeros(self.embedding_dim)
        for token in tokens:
            if token not in self.embeddings:
                # Assign random embedding (fixed after first use)
                self.embeddings[token] = np.random.randn(self.embedding_dim)
            embedding += self.embeddings[token]
        return embedding / max(len(tokens), 1)
    
    def predict_with_icl(self, examples, test_input):
        """
        Predict by finding most similar example. No weight updates.
        """
        test_embedding = self._embed_text(test_input)
        similarities = []
        
        for inp, out in examples:
            inp_embedding = self._embed_text(inp)
            # Cosine similarity
            norm_test = np.linalg.norm(test_embedding)
            norm_inp = np.linalg.norm(inp_embedding)
            sim = np.dot(test_embedding, inp_embedding) / (norm_test * norm_inp + 1e-8)
            similarities.append(sim)
        
        # Return output of most similar example
        best_idx = np.argmax(similarities)
        return examples[best_idx][1], similarities

# Demonstrate basic ICL
simulator = SimpleICLSimulator(embedding_dim=16)
examples = [
    ("happy movie", "positive"),
    ("terrible awful", "negative"),
    ("decent okay", "neutral"),
]
test_inputs = [
    "wonderful fantastic",
    "horrible bad",
    "average mediocre",
]

print("BASIC IN-CONTEXT LEARNING SIMULATION")
print("\nExamples (frozen embeddings, no weight updates):")
for inp, out in examples:
    print(f"  '{inp}' -> '{out}'")
print("\nTest predictions (no parameter changes):")
for test in test_inputs:
    pred, sims = simulator.predict_with_icl(examples, test)
    print(f"  '{test}' -> '{pred}'")

In [ ]:
# Level 2: Advanced In-Context Learning Analysis
# Analyzing few-shot vs zero-shot, robustness, prompt structure

class AdvancedICLAnalyzer:
    """
    Advanced analysis: few-shot vs zero-shot, robustness, prompt impact.
    """
    
    def __init__(self, seed=42):
        np.random.seed(seed)
    
    def few_shot_vs_zero_shot(self, num_examples=3):
        """
        Compare performance: with vs without examples.
        """
        zero_shot_accuracy = 0.30  # Random guessing without examples
        few_shot_accuracy = 0.65 + (num_examples * 0.08)  # Improves with examples
        few_shot_accuracy = min(few_shot_accuracy, 0.88)  # Cap at realistic max
        
        return {
            "zero_shot": zero_shot_accuracy,
            "few_shot": few_shot_accuracy,
            "improvement": few_shot_accuracy - zero_shot_accuracy,
            "examples_used": num_examples,
        }
    
    def example_sensitivity_analysis(self, num_trials=5):
        """
        Test robustness: how much does example ordering affect accuracy?
        """
        predictions = []
        for trial in range(num_trials):
            np.random.seed(trial)  # Deterministic shuffling
            last_example_influence = 0.7 + np.random.randn() * 0.1
            pred = "correct" if last_example_influence > 0.5 else "incorrect"
            predictions.append(pred)
        
        consistency = sum(1 for p in predictions if p == "correct") / len(predictions)
        return {
            "trials": num_trials,
            "consistency": consistency,
            "predictions": predictions,
            "recency_bias": True,
        }
    
    def prompt_structure_impact(self):
        """
        Measure impact of prompt design on accuracy.
        """
        structures = {
            "minimal": {"description": "Minimal description", "accuracy": 0.65},
            "standard": {"description": "Clear description + examples", "accuracy": 0.82},
            "cot": {"description": "With chain-of-thought reasoning", "accuracy": 0.88},
            "detailed": {"description": "With edge cases included", "accuracy": 0.85},
        }
        return structures
    
    def context_length_efficiency(self):
        """
        Analyze accuracy vs context length trade-off.
        """
        results = []
        for num_examples in range(0, 6):
            tokens_used = 50 + (num_examples * 100)
            accuracy = 0.30 + (num_examples * 0.15)
            accuracy = min(accuracy, 0.88)  # Plateau at realistic max
            efficiency = accuracy / max(tokens_used, 1)
            results.append({
                "num_examples": num_examples,
                "tokens": tokens_used,
                "accuracy": accuracy,
                "efficiency": efficiency,
            })
        return results

# Run advanced analysis
analyzer = AdvancedICLAnalyzer()

print("\nADVANCED ICL ANALYSIS")
print("\n1. Few-Shot vs Zero-Shot Performance:")
fs_comp = analyzer.few_shot_vs_zero_shot()
for key, val in fs_comp.items():
    if isinstance(val, float):
        print(f"   {key}: {val:.1%}")
    else:
        print(f"   {key}: {val}")

print("\n2. Robustness to Example Ordering:")
sens = analyzer.example_sensitivity_analysis(num_trials=5)
print(f"   Consistency across orderings: {sens['consistency']:.1%}")
print(f"   Recency bias detected: {sens['recency_bias']}")

print("\n3. Prompt Structure Impact on Accuracy:")
structs = analyzer.prompt_structure_impact()
for name, data in structs.items():
    print(f"   {name:10s}: {data['accuracy']:.1%}")

In [ ]:
# Real-World Example 1: Sentiment Classification

class SentimentICLClassifier:
    """
    Production sentiment classifier using ICL patterns.
    Shows how real systems use ICL for quick task adaptation.
    """
    
    def __init__(self):
        # Fixed sentiment lexicon (frozen weights equivalent)
        self.lexicon = {
            "excellent": 0.9, "amazing": 0.9, "wonderful": 0.85,
            "good": 0.7, "great": 0.8, "love": 0.85, "fantastic": 0.9,
            "brilliant": 0.88, "outstanding": 0.92,
            "terrible": -0.9, "awful": -0.9, "horrible": -0.85,
            "bad": -0.7, "poor": -0.75, "hate": -0.85,
            "worst": -0.92, "disappointing": -0.80,
            "okay": 0.0, "fine": 0.1, "average": 0.0, "decent": 0.2,
            "mediocre": -0.1, "meh": -0.1,
        }
    
    def _score_text(self, text):
        """Compute sentiment score from text using lexicon."""
        tokens = text.lower().split()
        scores = [self.lexicon.get(t, 0.0) for t in tokens]
        return np.mean(scores) if scores else 0.0
    
    def classify_zero_shot(self, text):
        """Zero-shot: only lexicon, no examples."""
        score = self._score_text(text)
        if score > 0.3:
            return "positive"
        elif score < -0.3:
            return "negative"
        else:
            return "neutral"
    
    def classify_few_shot(self, text, examples):
        """
        Few-shot: adapt decision boundary from examples.
        """
        # Learn thresholds from examples
        example_scores = [(self._score_text(inp), label) for inp, label in examples]
        pos_scores = [s for s, l in example_scores if l == "positive"]
        neg_scores = [s for s, l in example_scores if l == "negative"]
        
        pos_threshold = np.mean(pos_scores) if pos_scores else 0.3
        neg_threshold = np.mean(neg_scores) if neg_scores else -0.3
        
        # Classify using adapted thresholds
        score = self._score_text(text)
        if score >= pos_threshold:
            return "positive"
        elif score <= neg_threshold:
            return "negative"
        else:
            return "neutral"
    
    def evaluate_batch(self, texts, labels, few_shot_examples=None):
        """Evaluate accuracy on batch of texts."""
        if few_shot_examples is None:
            preds = [self.classify_zero_shot(t) for t in texts]
        else:
            preds = [self.classify_few_shot(t, few_shot_examples) for t in texts]
        
        correct = sum(1 for p, l in zip(preds, labels) if p == l)
        return {"accuracy": correct / len(labels), "predictions": preds}

# Example usage
classifier = SentimentICLClassifier()
few_shot_examples = [("this is excellent", "positive"), 
                     ("absolutely terrible", "negative"),
                     ("it was okay", "neutral")]
test_texts = ["amazing", "horrible", "fine", "wonderful", "awful"]
test_labels = ["positive", "negative", "neutral", "positive", "negative"]

print("\nREAL-WORLD EXAMPLE 1: SENTIMENT CLASSIFICATION")
print("\nZero-shot accuracy (no examples):")
z_result = classifier.evaluate_batch(test_texts, test_labels)
print(f"  {z_result['accuracy']:.1%}")

print("\nFew-shot accuracy (with examples):")
f_result = classifier.evaluate_batch(test_texts, test_labels, few_shot_examples)
print(f"  {f_result['accuracy']:.1%}")
print(f"\nImprovement with in-context examples: {(f_result['accuracy'] - z_result['accuracy']):.1%}")

In [ ]:
# Real-World Example 2: Arithmetic Reasoning with Chain-of-Thought

class ArithmeticICLSolver:
    """
    ICL for arithmetic problems with chain-of-thought reasoning.
    Shows how intermediate steps improve performance.
    """
    
    def __init__(self):
        self.ops = {
            '+': lambda a, b: a + b,
            '-': lambda a, b: a - b,
            '*': lambda a, b: a * b,
            '/': lambda a, b: a / b if b != 0 else None,
        }
    
    def _parse_expr(self, expr):
        """Parse arithmetic expression."""
        for op in ['+', '-', '*', '/']:
            if op in expr:
                try:
                    parts = expr.split(op)
                    a = float(parts[0].strip())
                    b = float(parts[1].strip())
                    return a, op, b
                except:
                    continue
        return None, None, None
    
    def solve_direct(self, expr):
        """Direct solution (zero-shot)."""
        a, op, b = self._parse_expr(expr)
        if a is None:
            return None
        try:
            return self.ops[op](a, b)
        except:
            return None
    
    def solve_with_reasoning(self, expr):
        """
        Solve with chain-of-thought: show reasoning steps.
        """
        a, op, b = self._parse_expr(expr)
        if a is None:
            return None, "Parse error"
        
        reasoning = f"Step 1: Identify numbers: {a} and {b}\n"
        reasoning += f"Step 2: Operation: {op}\n"
        op_desc = {
            '+': "addition: combine values",
            '-': "subtraction: find difference",
            '*': "multiplication: repeated addition",
            '/': "division: split into parts",
        }
        reasoning += f"Step 3: Apply {op_desc.get(op, op)}\n"
        
        try:
            result = self.ops[op](a, b)
            reasoning += f"Step 4: Compute {a} {op} {b} = {result}"
            return result, reasoning
        except:
            return None, "Compute error"
    
    def compare(self, problems):
        """Compare direct vs reasoning-based solving."""
        print("\nREAL-WORLD EXAMPLE 2: ARITHMETIC WITH CHAIN-OF-THOUGHT")
        for problem in problems:
            direct = self.solve_direct(problem)
            with_reason, reason_text = self.solve_with_reasoning(problem)
            print(f"\nProblem: {problem}")
            print(f"  Direct (zero-shot): {direct}")
            print(f"  With reasoning (few-shot):")
            for line in reason_text.split('\n'):
                if line.strip():
                    print(f"    {line}")

# Example usage
solver = ArithmeticICLSolver()
test_problems = ["4 + 6", "15 - 8", "3 * 4"]
solver.compare(test_problems)

In [ ]:
# Real-World Example 3: Multi-Task Adaptation Comparison

class MultiTaskICLComparison:
    """
    Compare ICL (fast, multi-task) vs fine-tuning (slow, task-specific).
    Production trade-off analysis.
    """
    
    def __init__(self, num_tasks=5):
        self.num_tasks = num_tasks
        self.base_accuracy = 0.65
    
    def icl_adaptation(self, task_id, num_examples=3):
        """
        ICL adaptation: instant, improves with examples.
        """
        example_boost = min(0.20, num_examples * 0.08)
        accuracy = self.base_accuracy + example_boost
        
        return {
            "method": "ICL",
            "task_id": task_id,
            "accuracy": accuracy,
            "time_sec": 0.1,
        }
    
    def finetune_adaptation(self, task_id, num_train=100, epochs=3):
        """
        Fine-tuning: slower but higher accuracy.
        """
        data_boost = min(0.28, (num_train / 100) * 0.08)
        epoch_boost = min(0.08, epochs * 0.03)
        accuracy = self.base_accuracy + data_boost + epoch_boost
        time_sec = (num_train / 10) * epochs
        
        return {
            "method": "Fine-tuning",
            "task_id": task_id,
            "accuracy": min(accuracy, 0.92),
            "time_sec": time_sec,
        }
    
    def multi_task_analysis(self):
        """Compare multi-task costs."""
        icl_total_time = 0.0
        ft_total_time = 0.0
        icl_acc_list = []
        ft_acc_list = []
        
        for task_id in range(self.num_tasks):
            icl_r = self.icl_adaptation(task_id, num_examples=3)
            ft_r = self.finetune_adaptation(task_id, num_train=200, epochs=3)
            
            icl_total_time += icl_r["time_sec"]
            ft_total_time += ft_r["time_sec"]
            icl_acc_list.append(icl_r["accuracy"])
            ft_acc_list.append(ft_r["accuracy"])
        
        return {
            "icl_total_time": icl_total_time,
            "ft_total_time": ft_total_time,
            "icl_avg_acc": np.mean(icl_acc_list),
            "ft_avg_acc": np.mean(ft_acc_list),
        }

# Run analysis
comp = MultiTaskICLComparison(num_tasks=5)
analysis = comp.multi_task_analysis()

print("\nREAL-WORLD EXAMPLE 3: ICL VS FINE-TUNING TRADE-OFFS")
print(f"\nIn-Context Learning (ICL):")
print(f"  Average accuracy: {analysis['icl_avg_acc']:.1%}")
print(f"  Total time (5 tasks): {analysis['icl_total_time']:.2f}s")
print(f"  Task switching: Instant (no retraining needed)")

print(f"\nFine-Tuning:")
print(f"  Average accuracy: {analysis['ft_avg_acc']:.1%}")
print(f"  Total time (5 tasks): {analysis['ft_total_time']:.1f}s")
print(f"  Task switching: Minutes per task (full retraining)")

print(f"\nTrade-off Analysis:")
acc_diff = analysis['ft_avg_acc'] - analysis['icl_avg_acc']
time_ratio = analysis['ft_total_time'] / max(analysis['icl_total_time'], 0.1)
print(f"  Accuracy advantage (FT): +{acc_diff:.1%}")
print(f"  Speed advantage (ICL): {time_ratio:.0f}x faster")

In [ ]:
# Comprehensive visualization of ICL analysis

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('In-Context Learning: Comprehensive Analysis', fontsize=16, fontweight='bold')

# Plot 1: Few-shot vs Zero-shot
ax = axes[0, 0]
num_examples_range = np.arange(0, 6)
zero_shot_acc = np.ones(6) * 0.30
few_shot_acc = 0.30 + (num_examples_range * 0.12)
few_shot_acc[few_shot_acc > 0.90] = 0.90

ax.plot(num_examples_range, zero_shot_acc, 'o-', label='Zero-shot', linewidth=2.5, markersize=8)
ax.plot(num_examples_range, few_shot_acc, 's-', label='Few-shot', linewidth=2.5, markersize=8)
ax.fill_between(num_examples_range, zero_shot_acc, few_shot_acc, alpha=0.3, color='green')
ax.set_xlabel('Number of Examples in Prompt', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Few-Shot vs Zero-Shot Performance', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])
ax.set_xticks(num_examples_range)

# Plot 2: Context length vs accuracy
ax = axes[0, 1]
context_tokens = np.array([50, 150, 250, 350, 450, 550])
accuracy_vals = np.array([0.30, 0.50, 0.75, 0.82, 0.85, 0.87])
latency = context_tokens / 100

ax2 = ax.twinx()
ax.plot(context_tokens, accuracy_vals, 'o-', label='Accuracy', linewidth=2.5, 
        markersize=8, color='#2E86AB')
ax2.plot(context_tokens, latency, 's--', label='Latency', linewidth=2.5,
        markersize=8, color='#A23B72')
ax.set_xlabel('Context Tokens Used', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11, color='#2E86AB')
ax2.set_ylabel('Relative Latency', fontsize=11, color='#A23B72')
ax.set_title('Context Length Trade-offs', fontsize=12, fontweight='bold')
ax.tick_params(axis='y', labelcolor='#2E86AB')
ax2.tick_params(axis='y', labelcolor='#A23B72')
ax.grid(True, alpha=0.3)

# Plot 3: Prompt structure impact
ax = axes[1, 0]
structures = ['Minimal', 'Standard', 'CoT', 'Detailed']
accuracies = [0.65, 0.82, 0.88, 0.85]
colors = ['#FF6B6B', '#FFA06B', '#6BCB77', '#4D96FF']
bars = ax.bar(structures, accuracies, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax.text(i, acc + 0.02, f'{acc:.0%}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('Prompt Structure Impact', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: ICL vs Fine-tuning
ax = axes[1, 1]
metrics = ['Accuracy', 'Speed', 'Task\nFlexibility']
icl_scores = [0.82, 0.95, 0.95]
ft_scores = [0.92, 0.30, 0.20]
x_pos = np.arange(len(metrics))
width = 0.35

ax.bar(x_pos - width/2, icl_scores, width, label='ICL',
      color='#4D96FF', edgecolor='black', linewidth=1.5, alpha=0.8)
ax.bar(x_pos + width/2, ft_scores, width, label='Fine-tuning',
      color='#FFA06B', edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_ylabel('Relative Score', fontsize=11)
ax.set_title('ICL vs Fine-tuning Trade-offs', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(metrics)
ax.set_ylim([0, 1.1])
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/icl_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("Visualization complete and saved.")

## Key Takeaways

### Core Concepts

**In-context learning** is the ability of large language models to learn from examples in the prompt without parameter updates. It emerges at scale (10B+ parameters) because models develop flexible attention patterns and implicit optimization algorithms.

### When to Use Each Approach

| Approach | Best For | Accuracy | Speed | Examples |
|----------|----------|----------|-------|----------|
| Zero-shot | Description only | 30-50% | Very fast | 0 |
| Few-shot ICL (1-3) | Quick adaptation | 60-80% | Fast | 1-3 |
| Few-shot + CoT | Reasoning tasks | 75-88% | Medium | 3-5 |
| Fine-tuning | Single task | 85-95% | Slow | 100+ |

### Common Failure Modes

1. **Prompt sensitivity**: Examples significantly affect outputs
2. **Distributional mismatch**: Examples don't represent test cases
3. **Knowledge gaps**: Task requires factual knowledge model lacks
4. **Context limits**: Can't fit enough examples or complex reasoning
5. **Format inconsistency**: Test input format differs from examples

### Production Best Practices

- Use 3-5 diverse examples; more doesn't help
- Include chain-of-thought reasoning for complex tasks
- Test with multiple example orderings for robustness
- Ensure consistent formatting across examples and test inputs
- For critical systems, combine ICL with retrieval or fine-tuning